In [ ]:
# ─────────────────────────────────────────────────────────────
# خلية 0 (الأولى): تفعيل التشخيص المتزامن لأخطاء CUDA
# ─────────────────────────────────────────────────────────────
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'   # يجعل أخطاء CUDA تظهر فورًا في مكان وقوعها

In [ ]:
# ─────────────────────────────────────────────────────────────
# خلية 1: تثبيت المكتبات واستيرادها وضبط البذور
# ─────────────────────────────────────────────────────────────
!pip install -q kagglehub transformers torch scikit-learn tensorboard tqdm

import random
import zipfile
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, RobertaConfig, RobertaForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
import kagglehub

# ضبط البذور
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
torch.backends.cudnn.benchmark = True


In [ ]:
# ─────────────────────────────────────────────────────────────
# خلية 2: تحميل البيانات وفك الضغط
# ─────────────────────────────────────────────────────────────
path = kagglehub.dataset_download("fizzbuzz/cleaned-toxic-comments")

if "archive.zip" in os.listdir(path):
    with zipfile.ZipFile(os.path.join(path, "archive.zip"), "r") as z:
        z.extractall(path)

train_csv = os.path.join(path, "train_preprocessed.csv")
df = pd.read_csv(train_csv)
df.rename(columns={"severe_tox": "severe_toxic"}, inplace=True)

label_cols = ["identity_hate", "insult", "obscene", "severe_toxic", "threat", "toxic"]
train_full = df[df.set == "train"].reset_index(drop=True)
train_df    = train_full.sample(frac=0.8, random_state=42)
val_df      = train_full.drop(train_df.index).reset_index(drop=True)


In [ ]:
# ─────────────────────────────────────────────────────────────
# خلية 3: تعريف Dataset و DataLoader
# ─────────────────────────────────────────────────────────────
class ToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len: int = 256):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

MODEL_NAME  = "roberta-base"
BATCH_SIZE  = 256    # كما طلبت
NUM_WORKERS = max(1, os.cpu_count() - 1)

tokenizer   = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds    = ToxicDataset(train_df.comment_text.values, train_df[label_cols].values, tokenizer)
val_ds      = ToxicDataset(val_df.comment_text.values,   val_df[label_cols].values,   tokenizer)

train_loader= DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=NUM_WORKERS, pin_memory=True)
val_loader  = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)


In [ ]:
# ─────────────────────────────────────────────────────────────
# خلية 4: إعداد النموذج من الصفر
# ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = RobertaConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_size=512,
    num_hidden_layers=8,
    num_attention_heads=8,
    intermediate_size=2048,
    num_labels=len(label_cols),
    max_position_embeddings=512,
    problem_type="multi_label_classification",
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
)

model     = RobertaForSequenceClassification(config).to(device)
optimizer = AdamW(model.parameters(), lr=4e-5, weight_decay=1e-2)

EPOCHS      = 5
total_steps = len(train_loader) * EPOCHS
warmup_steps= int(0.1 * total_steps)
scheduler   = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

pos_weights = []
for c in label_cols:
    vals = train_df[c].astype(int).values
    neg, pos = (vals == 0).sum(), (vals == 1).sum()
    pos_weights.append(neg / (pos + 1e-10))
pos_weights = torch.tensor(pos_weights, dtype=torch.float).to(device)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weights)
scaler    = GradScaler()


In [ ]:
# ─────────────────────────────────────────────────────────────
# خلية 5: TensorBoard و Early Stopping
# ─────────────────────────────────────────────────────────────
writer        = SummaryWriter("runs/toxic_custom_roberta")
best_val_loss = float("inf")
patience, counter, gstep = 3, 0, 0


In [ ]:
# ─────────────────────────────────────────────────────────────
# خلية 6: Training loop مع فحص قيم input_ids
# ─────────────────────────────────────────────────────────────
vocab_sz = model.config.vocab_size

for epoch in range(1, EPOCHS + 1):
    print(f"\n▶️ Epoch {epoch}/{EPOCHS}")
    model.train()
    train_loss = 0.0

    for batch in tqdm(train_loader, desc="📦 Training", leave=False):
        optimizer.zero_grad()
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        # 🔍 Debug: تأكد من أن جميع input_ids < vocab_size
        max_id = input_ids.max().item()
        if max_id >= vocab_sz:
            raise ValueError(f"Found input_id {max_id} ≥ vocab_size {vocab_sz}")

        with autocast():
            logits = model(input_ids, attention_mask=attention_mask).logits
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item()
        gstep += 1
        if gstep % 100 == 0:
            writer.add_scalar("Train/Loss", loss.item(), gstep)
            writer.add_scalar("Train/LR", scheduler.get_last_lr()[0], gstep)

    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []

    for batch in tqdm(val_loader, desc="🔍 Validation", leave=False):
        with torch.no_grad(), autocast():
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            if input_ids.max().item() >= vocab_sz:
                raise ValueError(f"Found input_id ≥ vocab_size during validation")

            logits   = model(input_ids, attention_mask=attention_mask).logits
            val_loss+= criterion(logits, labels).item()

            probs    = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(probs)
            all_labels.append(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    all_preds    = np.vstack(all_preds)
    all_labels   = np.vstack(all_labels)
    roc_auc      = roc_auc_score(all_labels, all_preds, average="macro")
    preds_bin    = (all_preds > 0.5).astype(int)
    prec, recall, f1, _ = precision_recall_fscore_support(
        all_labels, preds_bin, average="macro", zero_division=0
    )

    writer.add_scalar("Val/Loss",     avg_val_loss, epoch)
    writer.add_scalar("Val/ROC_AUC",  roc_auc,       epoch)
    writer.add_scalar("Val/F1",       f1,            epoch)

    print(
        f"Train Loss={avg_train_loss:.4f} | "
        f"Val Loss={avg_val_loss:.4f} | "
        f"ROC-AUC={roc_auc:.4f} | F1={f1:.4f}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter       = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict()
        }, "best_model.pt")
        print("💾 تم حفظ أفضل نموذج!")
    else:
        counter += 1
        if counter >= patience:
            print("🛑 تم تفعيل الإيقاف المبكر!")
            break


In [ ]:
# ─────────────────────────────────────────────────────────────
# خلية 7: حفظ النموذج والـ Tokenizer والاختبار السريع
# ─────────────────────────────────────────────────────────────
model.save_pretrained("toxic_roberta_model")
tokenizer.save_pretrained("toxic_roberta_tokenizer")
writer.close()

# اختبارٍ سريع
checkpoint = torch.load("best_model.pt")
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Illustrative input and individual predictions omitted from publication copy.


Historical cell 8 contained ten illustrative comments and individual predictions. It has been omitted from this publication copy. The unchanged original remains in the private recovery package. These examples were not a labeled test set.